In [ ]:
import math
import os

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import polars as pl

In [ ]:
# データファイルがkaggle上のどこにあるか確認
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# 1. データの読み込み

In [ ]:
# 元データが5GBなので、メモリ対策としてLazyFrameとして定義
fp_2019_oct = "/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/2019-Oct.csv"
lazy_df_2019_oct = pl.scan_csv(fp_2019_oct)

In [ ]:
# 元データが9GBなので、メモリ対策としてLazyFrameとして定義

fp_2019_nov = "/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/2019-Nov.csv"
lazy_df_2019_nov = pl.scan_csv(fp_2019_nov)

# 2. データの中身の確認

In [ ]:
# 列名とデータ型の確認
lazy_df_2019_oct.collect_schema()

In [ ]:
# 列名とデータ型の確認
lazy_df_2019_nov.collect_schema()

In [ ]:
# 全体の行数確認

len_rows_oct = lazy_df_2019_oct.select(pl.len()).collect().item()
len_rows_nov = lazy_df_2019_nov.select(pl.len()).collect().item()

print(f"Oct:{len_rows_oct}\nNov:{len_rows_nov}")

# ↓結果
# Oct:42448764
# Nov:67501979

## 2.1 欠損確認

In [ ]:
lazy_df_2019_oct.null_count().collect()

# ↓結果
# event_time	event_type	product_id	category_id	category_code	brand	price	user_id	user_session
# u32	u32	u32	u32	u32	u32	u32	u32	u32
# 0	0	0	0	13515609	6113008	0	0	2

# 今回のファネル分析で使用するuser_id、event_typeには欠損なし
# category_idにも欠損がないので、どのカテゴリーが購入まで行くのに有効かまで調査可能。
# category_codeは13515609/42448764が欠損している。
# user_sessionの欠損が２件あり欠損列をdropしても問題ないが、同一セッション内での行動を追わずに、
# 今回はユニークなuser_idの数で判定する。もし時系列情報を使うとしたら、欠損のないevent_timeで大まかに把握する程度にとどめる。

In [ ]:
lazy_df_2019_nov.null_count().collect()

# ↓結果
# event_time	event_type	product_id	category_id	category_code	brand	price	user_id	user_session
# u32	u32	u32	u32	u32	u32	u32	u32	u32
# 0	0	0	0	21898171	9218235	0	0	10

## 2.2 event_typeの確認

In [ ]:
# ファネル分析に必須な、ユーザーの行動の種類を確認

lazy_unique_oct = lazy_df_2019_oct.select(pl.col("event_type").unique())
df_unique_event_type_oct = lazy_unique_oct.collect()


In [ ]:
df_unique_event_type_oct
# ↓event_typeのユニーク値の結果
# event_type
# str
# "view"
# "purchase"
# "cart"

In [ ]:
# それぞれの数を確認
lazy_df_2019_oct.select(pl.col("event_type").value_counts()).collect()

# ↓結果
# shape: (3, 1)
# event_type
# struct[2]
# {"purchase",742849}
# {"view",40779399}
# {"cart",926516}

In [ ]:
lazy_unique_nov = lazy_df_2019_nov.select(pl.col("event_type").unique())
df_unique_event_type_nov = lazy_unique_nov.collect()

In [ ]:
df_unique_event_type_nov

# event_type
# str
# "purchase"
# "view"
# "cart"

In [ ]:
# それぞれの数を確認
lazy_df_2019_nov.select(pl.col("event_type").value_counts()).collect()

# ↓結果
# {"purchase",916939}
# {"view",63556110}
# {"cart",3028930}

## 2.3 category_codeの確認

In [ ]:
df_category_code_nov = lazy_df_2019_nov.select(pl.col("category_code").unique()).collect()

In [ ]:
df_category_code_nov.head(5)

In [ ]:
df_category_code_oct = lazy_df_2019_oct.select(pl.col("category_code").unique()).collect()


In [ ]:
df_category_code_oct.head(5)

## 2.4 user_idのユニーク数を確認

In [ ]:
# 元データが合計で１億行以上で約14GBあるので、ランダムでuser_idを抽出してファネル分析につなげる
# そのために、user_idのユニーク数を把握する

df_oct_unique_user_id = lazy_df_2019_oct.select(pl.col("user_id").unique()).collect()
print("Oct ユニークuser_id抽出完了")

df_nov_unique_user_id = lazy_df_2019_nov.select(pl.col("user_id").unique()).collect()
print("Nov ユニークuser_id抽出完了")

df_combined = pl.concat([df_oct_unique_user_id, df_nov_unique_user_id])
print("concat 完了")

# ユニークなuser_idの一覧を作成
df_final_unique_user_id = df_combined.unique()

In [ ]:
len(df_final_unique_user_id)

# user_idのユニーク数
# →5316649

# 3. ファネル分析

なお、本分析でのファネル分析は、**各イベントを経験したユニークユーザー数によるファネル**と定義する。

## 3.1 view→cart→purchaseのuniqueなuser_id数の推移

In [ ]:
use_cols = ["user_id","event_type"]

In [ ]:
# 元データが非常に大きいので、一部のuser_idを指定して、抽出してから分析を行う
# 一旦1000ユーザーでだいたいの値を見てみる。
target_uid = df_final_unique_user_id.sample(n=1000, seed=0)

In [ ]:
target_uid["user_id"].head(5).to_list()

In [ ]:
lazy_df_2019_merged = pl.concat([
    lazy_df_2019_oct.select(use_cols).filter(pl.col("user_id").is_in(target_uid["user_id"].to_list())),
    lazy_df_2019_nov.select(use_cols).filter(pl.col("user_id").is_in(target_uid["user_id"].to_list()))
])


In [ ]:
(
    lazy_df_2019_merged.filter(pl.col("user_id").is_in(target_uid["user_id"].to_list()))
    .group_by("event_type")
    .agg(pl.col("user_id").n_unique())
    .collect()
).transpose(include_header=True, column_names="event_type")


# ↓1000ユーザーの結果
# column	view	cart	purchase
# str	u32	u32	u32
# "user_id"	1000	200	133
# 離脱者数は、800 67
# 離脱率だと、0.8 0.335


In [ ]:
# 1000ユーザーでの、cart、purchase間の離脱率の誤差は、±6.5%以内
1.96 * math.sqrt((0.335*(1-0.335)/200))

In [ ]:
# 母数が一番少ないcart > purchaseの確率を求める際に、誤差は2%以内と考えられる母数を概算する
# 離脱率は1000ユーザーで計算したものを用いるため、この値に起因する誤差が発生する点には留意する

# ランダムサンプリングなので、正規分布を仮定する
# 95％信頼区間の想定で、z = 1.96とする

# 0.02 = 1.96 * math.sqrt(0.335*(1-0.335)/x)
# (0.02) ** 2 = (1.96)**2 * ((0.335*(1-0.335))/x)
# x = (1.96)**2 * ((0.335*(1-0.335))) / ((0.02) ** 2)

(1.96)**2 * ((0.335*(1-0.335))) / ((0.02) ** 2)
# ↓結果
# 2139.5310999999997

# キリよく2150とすると、
(2150/133) * 1000

# ↓結果
# 16165.413533834588
# となるので、概算ではあるが、誤差が±2%になるだろうと見なすために、
# 少し多めにキリがよく16500のユニークユーザーを抽出する

In [ ]:
sampled_user_ids = df_final_unique_user_id.sample(n=16500, seed=0)

In [ ]:
lazy_df_2019_merged_n16500 = pl.concat([
    lazy_df_2019_oct.select(use_cols).filter(pl.col("user_id").is_in(sampled_user_ids["user_id"].to_list())),
    lazy_df_2019_nov.select(use_cols).filter(pl.col("user_id").is_in(sampled_user_ids["user_id"].to_list()))
])

# ２つのlazydf両方に入っているユークユーザーは重複するので、さらにユニークをとる必要が残っている

In [ ]:
df_n16500 = (
    lazy_df_2019_merged_n16500.filter(pl.col("user_id").is_in(sampled_user_ids["user_id"].to_list()))
    .group_by("event_type")
    .agg(pl.col("user_id").n_unique())
    .collect()
).transpose(include_header=True, column_names="event_type")

In [ ]:
df_n16500

# ↓結果の確認
# shape: (1, 4)
# column	purchase	cart	view
# str	u32	u32	u32
# "user_id"	2148	3302	16497

# view>cartの離脱率
# 0.7998

# cart>purchaseの離脱率
# 0.3494

In [ ]:
df_long = df_n16500.unpivot(
    index="column",
    on=["view", "cart", "purchase"],
    variable_name="stage",
    value_name="count",
)

In [ ]:
df_long

In [ ]:
pio.renderers.default = "iframe"

fig = px.funnel(
    df_long,
    x="stage",  
    y="count", 
    title="コンバージョンファネル",
)
fig.update_yaxes(autorange="reversed")
# fig.write_html("/kaggle/working/funnel_chart.html")

# fig.write_image("/kaggle/working/funnel_chart.png", scale=2)
# 
# write_image→pip install -U kaleidoが必要だけど、kaggle notebookのデフォルト環境が壊れないか
# 調べるまで保留

fig.show()

## 3.2 カテゴリー別の離脱率

### 3.2.1 category_codeの抽出

In [ ]:
df_category_code = pl.concat([df_category_code_oct, df_category_code_nov], how="vertical")

In [ ]:
sr_category_code = df_category_code["category_code"].unique()

# category_codeの欠損は一定量あるが、今回は簡易的にカテゴリごとにファネルを見るため、
# 数値の羅列のcategory_idではなく、直感的に意味がわかるcategory_codeを使用する。
# 欠損のないcategory_idを用い、category_idとcategory_codeを紐づけるやり方もあるが、
# 今回は簡易的に分析を行う

In [ ]:
sr_category_code

In [ ]:
sr_unique_category_code = sr_category_code.str.split(by=".").list.get(0).unique()

### 3.2.2 カテゴリごとにユニークユーザー数を求める

In [ ]:
use_cols_for_cat = ["user_id","event_type", "category_code"]

In [ ]:
lazy_df_2019_merged_n16500_with_cat = pl.concat([
    lazy_df_2019_oct.select(use_cols_for_cat).filter(pl.col("user_id").is_in(sampled_user_ids["user_id"].to_list())),
    lazy_df_2019_nov.select(use_cols_for_cat).filter(pl.col("user_id").is_in(sampled_user_ids["user_id"].to_list()))
])

In [ ]:
# 結果を入れるdf
df_funnel_by_category = pl.DataFrame()

# 結合用のリスト
dfs = [pl.DataFrame({"event_type":["view", "cart", "purchase"]})]

# categoryによっては、
# "view", "cart", "purchase"
# の３つの段階でuser_idがないカテゴリーがあるかもしれないので、
# concatの時のshapeのエラー対策で空のdfを作っておく
df_dummy = pl.DataFrame({
    # "category_code": [0,0,0],
    "event_type":["view", "cart", "purchase"],
    "dummy": [0,0,0],
})

# df_catがすでにある場合は、処理に時間がかかるので作り直さない
if "df_cat" not in locals():
    df_cat = lazy_df_2019_merged_n16500_with_cat.with_columns(
        pl.col("category_code")
        .fill_null("unknown.unknown")
        .str.split(".").list.get(0).alias("top_level_category")
    ).collect()


i=0
for category in sr_unique_category_code: 

    # 一旦欠損はスキップする
    # 後で、unknowとして処理するのもあり
    if category is None:
        continue
    
    print(category)    

    df = (
        df_cat.filter(pl.col("top_level_category") == category)
        .group_by("event_type")
        .agg(pl.col("user_id").n_unique())
    )

    df = df.with_columns(pl.col("user_id").alias(category))

    df = df_dummy.join(df, on="event_type", how="left")

    # print(df.select("event_type", category))

    dfs.append(df.select(category))

    # 動作確認用
    # if i > 2:
    #     break
    # i += 1
    
df_funnel_by_category = pl.concat(dfs, how="horizontal")

# メモリ対策で、使用したdf_catを削除
# del df_cat

In [ ]:
df_funnel_by_category
# .transpose(include_header=True, column_names="event_type")

In [ ]:
# csvで保存
(
    df_funnel_by_category
    .transpose(include_header=True, column_names="event_type")
    .write_csv("funnel_by_category.csv")
)

In [ ]:
df_eda = df_funnel_by_category.transpose(include_header=True, column_names="event_type")

In [ ]:
type(df_eda)

In [ ]:
df_eda = df_eda.to_pandas()

In [ ]:
df_eda["view2cart_drop_off"] = (df_eda["view"]-df_eda["cart"])/df_eda["view"]
df_eda["cart2purchase_drop_off"] = (df_eda["cart"]-df_eda["purchase"])/df_eda["cart"]

# TODO
# 汎用的なコードにする場合は、0除算の対応を入れる

In [ ]:
df_eda

# ↓結果
# 	column	view	cart	purchase	view2cart_drop_off	cart2purchase_drop_off
# 0	construction	714	55.0	28.0	0.922969	0.490909
# 1	electronics	9529	1965.0	1238.0	0.793787	0.369975
# 2	accessories	359	8.0	6.0	0.977716	0.250000
# 3	appliances	3400	519.0	331.0	0.847353	0.362235
# 4	apparel	1954	61.0	37.0	0.968782	0.393443
# 5	sport	257	7.0	4.0	0.972763	0.428571
# 6	computers	1633	189.0	113.0	0.884262	0.402116
# 7	medicine	22	2.0	NaN	0.909091	NaN
# 8	kids	734	32.0	17.0	0.956403	0.468750
# 9	auto	916	67.0	48.0	0.926856	0.283582
# 10	furniture	1383	65.0	40.0	0.953001	0.384615
# 11	country_yard	30	NaN	NaN	NaN	NaN
# 12	stationery	21	1.0	NaN	0.952381	NaN

In [ ]:
# メモ

# # 誤差を小さくするために、1000件を50回行い、平均の平均を求める。
# # >groupbyで出ないevent_typeが空になる

# # dfs = []

# for i in range(5):
#     sampled_user_ids = df_final_unique_user_id.sample(n=1000, seed=i)

#     print(sampled_user_ids.head(5))
    
#     df = (
#         lazy_df_2019_merged.filter(pl.col("user_id").is_in(sampled_user_ids["user_id"].to_list()))
#         .group_by("event_type")
#         .agg(pl.col("user_id").n_unique())
#         .collect()
#     )
    
#     if df.is_empty():
#         continue
        
#     df_t = df.transpose(include_header=True, column_names="event_type")
#     dfs.append(df_t)

#     print(f"Iteration: {i}")

# df_funnel = pl.concat(dfs)    
